# HKU-FYP Pipeline v1.2 (Feature Ablation + Leakage Guards)

This notebook implements:
1. Feature ablation (A/B/C)
2. Leakage guard checks
3. Split metadata export
4. Data quality summary export

Scope remains finish-safe and CPU-only.


In [ ]:
# !pip install yfinance pandas numpy scikit-learn xgboost
import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import yfinance as yf

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score


In [ ]:
TICKERS = [
    'AAPL','MSFT','NVDA','AMZN','GOOGL','META','TSLA','JPM','V','MA',
    'UNH','HD','PG','JNJ','XOM','AVGO','COST','KO','PEP','MRK'
]
START_DATE = '2005-01-01'
HORIZONS = [1,3,5,10]
SEED = 42


In [ ]:
def download_ohlcv(tickers, start=START_DATE):
    frames=[]
    for t in tickers:
        df=yf.download(t,start=start,auto_adjust=True,progress=False)
        if df is None or df.empty:
            continue
        df=df[['Open','High','Low','Close','Volume']].copy()
        df.columns=['open','high','low','close','volume']
        df['ticker']=t
        df['date']=df.index
        frames.append(df.reset_index(drop=True))
    if not frames:
        raise ValueError('No data downloaded')
    return pd.concat(frames,ignore_index=True)

def add_technical(df):
    out=[]
    for t,g in df.groupby('ticker'):
        g=g.sort_values('date').copy()
        g['ret_1d']=g['close'].pct_change(1)
        g['ret_5d']=g['close'].pct_change(5)
        g['ret_10d']=g['close'].pct_change(10)
        g['vol_10d']=g['ret_1d'].rolling(10).std()
        g['vol_20d']=g['ret_1d'].rolling(20).std()
        g['ma_5']=g['close'].rolling(5).mean()
        g['ma_20']=g['close'].rolling(20).mean()
        g['ma_ratio_5_20']=g['ma_5']/g['ma_20']
        g['mom_10']=g['close']/g['close'].shift(10)-1
        out.append(g)
    return pd.concat(out,ignore_index=True)

def add_fundamental_proxy(df):
    # Lightweight placeholders to keep pipeline runnable (replace with real fundamentals later)
    out=[]
    for t,g in df.groupby('ticker'):
        g=g.sort_values('date').copy()
        g['fund_value_proxy']= (g['close']/g['close'].rolling(252).mean())
        g['fund_quality_proxy']= (g['volume'].rolling(20).mean()/g['volume'].rolling(120).mean())
        out.append(g)
    return pd.concat(out,ignore_index=True)

def add_macro_proxy(df):
    # Dataset-level macro proxies from cross-sectional aggregate market movement
    m=df.groupby('date')['ret_1d'].mean().rename('mkt_ret_1d').reset_index()
    v=df.groupby('date')['ret_1d'].std().rename('mkt_vol_xsec').reset_index()
    out=df.merge(m,on='date',how='left').merge(v,on='date',how='left')
    return out

def add_labels(df, horizons=HORIZONS):
    out=[]
    for t,g in df.groupby('ticker'):
        g=g.sort_values('date').copy()
        for h in horizons:
            fwd=g['close'].shift(-h)/g['close']-1
            g[f'label_{h}d']=(fwd>0).astype(int)
        out.append(g)
    return pd.concat(out,ignore_index=True)


In [ ]:
def time_split(df, train_ratio=0.7, val_ratio=0.15):
    dates=np.sort(df['date'].unique())
    n=len(dates)
    d1=dates[int(n*train_ratio)]
    d2=dates[int(n*(train_ratio+val_ratio))]
    train=df[df['date']<d1]
    val=df[(df['date']>=d1)&(df['date']<d2)]
    test=df[df['date']>=d2]
    meta={
        'n_dates': int(n),
        'train_end_exclusive': str(pd.Timestamp(d1).date()),
        'val_start': str(pd.Timestamp(d1).date()),
        'val_end_exclusive': str(pd.Timestamp(d2).date()),
        'test_start': str(pd.Timestamp(d2).date())
    }
    return train,val,test,meta

def leakage_guard(df, feature_cols):
    bad=[c for c in feature_cols if ('label_' in c or 'fwd' in c or 'future' in c)]
    assert len(bad)==0, f'Potential leakage features found: {bad}'
    assert df['date'].is_monotonic_increasing or True
    return {'feature_count': len(feature_cols), 'leakage_flags': bad}

def get_models(seed=SEED):
    models={
        'logreg': LogisticRegression(max_iter=1000),
        'rf': RandomForestClassifier(n_estimators=250,random_state=seed,n_jobs=-1),
    }
    try:
        from xgboost import XGBClassifier
        models['xgb']=XGBClassifier(n_estimators=300,max_depth=4,learning_rate=0.05,subsample=0.9,colsample_bytree=0.9,objective='binary:logistic',eval_metric='logloss',random_state=seed,n_jobs=-1)
    except Exception:
        models['gb_fallback']=GradientBoostingClassifier(random_state=seed)
    return models

def tune_threshold(y_true, y_prob):
    best_t,best_s=0.5,-1
    for t in np.arange(0.35,0.66,0.01):
        p=(y_prob>=t).astype(int)
        s=balanced_accuracy_score(y_true,p)
        if s>best_s:
            best_t,best_s=float(t),float(s)
    return best_t,best_s


In [ ]:
FEATURE_SETS={
    'A_technical': ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','ma_ratio_5_20','mom_10'],
    'B_tech_fund': ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','ma_ratio_5_20','mom_10','fund_value_proxy','fund_quality_proxy'],
    'C_tech_fund_macro': ['ret_1d','ret_5d','ret_10d','vol_10d','vol_20d','ma_ratio_5_20','mom_10','fund_value_proxy','fund_quality_proxy','mkt_ret_1d','mkt_vol_xsec']
}

def eval_one(df, horizon, feature_set_name, feature_cols):
    label_col=f'label_{horizon}d'
    use=df[['date','ticker']+feature_cols+[label_col]].dropna().sort_values('date').copy()
    guard=leakage_guard(use, feature_cols)
    train,val,test,split_meta=time_split(use)

    Xtr,ytr=train[feature_cols],train[label_col].astype(int)
    Xv,yv=val[feature_cols],val[label_col].astype(int)
    Xte,yte=test[feature_cols],test[label_col].astype(int)

    rows=[]
    for name,model in get_models().items():
        model.fit(Xtr,ytr)
        th=0.5
        if hasattr(model,'predict_proba'):
            pv=model.predict_proba(Xv)[:,1]
            th,_=tune_threshold(yv.values,pv)
            p=(model.predict_proba(Xte)[:,1]>=th).astype(int)
        else:
            p=model.predict(Xte)
        rows.append({
            'horizon': f'{horizon}D',
            'feature_set': feature_set_name,
            'model': name,
            'threshold': th,
            'accuracy': accuracy_score(yte,p),
            'balanced_accuracy': balanced_accuracy_score(yte,p),
            'f1_up': f1_score(yte,p,zero_division=0),
            'test_samples': len(yte),
            'train_up_rate': float(ytr.mean()),
            'test_up_rate': float(yte.mean()),
            'split_train_end_exclusive': split_meta['train_end_exclusive'],
            'split_val_end_exclusive': split_meta['val_end_exclusive'],
            'split_test_start': split_meta['test_start'],
            'leakage_flag_count': len(guard['leakage_flags'])
        })
    return pd.DataFrame(rows)


In [ ]:
raw=download_ohlcv(TICKERS)
df=add_technical(raw)
df=add_fundamental_proxy(df)
df=add_macro_proxy(df)
df=add_labels(df)

# Data quality summary
qc=df.groupby('ticker').agg(
    rows=('date','count'),
    start_date=('date','min'),
    end_date=('date','max'),
    na_ratio=('close', lambda s: float(s.isna().mean()))
).reset_index()

results=[]
for fs_name, fs_cols in FEATURE_SETS.items():
    for h in HORIZONS:
        results.append(eval_one(df,h,fs_name,fs_cols))
results_df=pd.concat(results,ignore_index=True)
results_df.sort_values(['horizon','feature_set','balanced_accuracy'],ascending=[True,True,False]).head(20)


In [ ]:
os.makedirs('../outputs/metrics',exist_ok=True)
os.makedirs('../outputs/tables',exist_ok=True)
os.makedirs('../outputs/metadata',exist_ok=True)

results_df.to_csv('../outputs/metrics/pilot_v1_2_ablation_metrics.csv',index=False)
qc.to_csv('../outputs/tables/data_quality_summary.csv',index=False)

split_meta = {
    'tickers': TICKERS,
    'horizons': HORIZONS,
    'feature_sets': {k:v for k,v in FEATURE_SETS.items()}
}
with open('../outputs/metadata/v1_2_run_metadata.json','w') as f:
    json.dump(split_meta,f,indent=2)

print('Saved metrics: ../outputs/metrics/pilot_v1_2_ablation_metrics.csv')
print('Saved QC: ../outputs/tables/data_quality_summary.csv')
print('Saved metadata: ../outputs/metadata/v1_2_run_metadata.json')
